# 01. PEC Metrics on Synthetic Data

This notebook explains the metric logic on toy data. It is not the empirical dataset used in the paper.

<!-- PEC_NOTEBOOK_EXPLANATION_V1:metric_walkthrough_guide -->

## Reader guide: why synthetic data is used here

This notebook uses toy records so that each PEC metric can be inspected by hand. The goal is not to reproduce the paper values. The goal is to make the formulas intuitive before looking at the real six-month audit.

The paper uses the same metric families at scale:

- **Set overlap:** do stated categories overlap with clicked categories?
- **Distributional alignment:** do weighted preference distributions resemble clicked distributions?
- **Traceability:** can clicked items be found in logged recommendation lists?
- **Diversity and concentration:** is logged exposure broader or narrower than click consumption?

<!-- PEC_NOTEBOOK_EXPLANATION_V1:metric_definitions -->

## Metric definitions in plain language

Let $P_u$ be a user's stated preference set, $C_u$ the clicked categories, and $E_u$ the logged exposure items.

- **Jaccard overlap**: $|P_u \cap C_u| / |P_u \cup C_u|$. A high value means the stated preference set and clicked category set overlap strongly.
- **Divergence score**: $1 - \text{Jaccard}$. This turns overlap into a mismatch measure.
- **Cosine similarity**: compares two weighted distributions. In the paper this is useful because user profiles include probability-like preference weights, not only selected category names.
- **Normalized entropy**: measures how spread out a distribution is. Higher entropy means broader diversity.
- **HHI**: $\sum_j p_j^2$. Higher HHI means stronger concentration in fewer categories or publishers.
- **Top-share**: $\max_j p_j$. This is the share of the dominant category/publisher.
- **Jensen-Shannon divergence**: a bounded, symmetric distributional distance. It is useful when exposure and consumption are both distributions over the same category space.
- **Count matching**: samples exposure down to the number of clicks before comparing diversity. This prevents exposure from looking more diverse simply because there are many more exposure items than clicked items.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from pecgap.metrics import jaccard, normalized_entropy, hhi, top_share, js_divergence, count_matched_metric_gap
SYN = ROOT / 'data' / 'synthetic'

In [ ]:
profiles = pd.read_csv(SYN / 'user_profiles_clean.csv')
clicks = pd.read_csv(SYN / 'click_events_enriched.csv')
recs = pd.read_csv(SYN / 'recommendation_items_enriched.csv')
profiles, clicks, recs

## Preference-consumption alignment

For each user, compare stated category preferences with top clicked categories. A low Jaccard score means that stated preference and observed click consumption are not identical signals.

In [ ]:
u1_pref = ['politics', 'economy']
u1_click_top = ['politics', 'sports']
print('Jaccard:', jaccard(u1_pref, u1_click_top))
print('Divergence:', 1 - jaccard(u1_pref, u1_click_top))

## Diversity and concentration

Entropy is higher when labels are broader. HHI and top-share are higher when labels are more concentrated.

In [ ]:
broad = ['politics','economy','sports','technology']
narrow = ['politics','politics','politics','economy']
for name, values in [('broad', broad), ('narrow', narrow)]:
    print(name, {'entropy': normalized_entropy(values), 'hhi': hhi(values), 'top_share': top_share(values)})

## Jensen-Shannon divergence

JS divergence compares the whole exposure distribution with the click distribution. It is symmetric and remains finite when the two distributions have different supports.

In [ ]:
exposure = ['politics','economy','sports','technology','politics']
click = ['politics','politics','sports']
js_divergence(exposure, click)

## Count matching

Count matching samples exposure down to the number of observed clicks. This checks whether an entropy gap is merely a count artifact.

In [ ]:
count_matched_metric_gap(exposure, click, normalized_entropy, n_iter=1000, seed=7)